# Pizza Vision: deploy YOLO to Model Serving

Logs a PyFunc wrapper around `ultralytics.YOLO`, registers it to Unity
Catalog, and creates (or updates) a Model Serving endpoint that the
AppKit `serving` plugin consumes via the `detector` alias.

The endpoint accepts the standard MLflow Model Serving payload:

```json
{
  "dataframe_records": [
    {"image": "<base64-jpeg>", "conf": 0.35, "iou": 0.5}
  ]
}
```

and returns `{"predictions": [[{label, class_id, confidence, bbox}, ...]]}`
(one inner list per input row).

In [ ]:
dbutils.widgets.text("catalog", "users")
dbutils.widgets.text("schema", "lensiq")
dbutils.widgets.text("registered_name", "lensiq_detector")
dbutils.widgets.text("endpoint_name", "lensiq-detector")
dbutils.widgets.text("weights_url", "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
REGISTERED = f"{CATALOG}.{SCHEMA}.{dbutils.widgets.get('registered_name')}"
ENDPOINT = dbutils.widgets.get("endpoint_name")
WEIGHTS_URL = dbutils.widgets.get("weights_url")

In [ ]:
%pip install -q ultralytics==8.3.0 mlflow>=2.13 pillow numpy
dbutils.library.restartPython()

In [ ]:
import logging
import os
import urllib.request

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOG = logging.getLogger("deploy_yolo")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
REGISTERED = f"{CATALOG}.{SCHEMA}.{dbutils.widgets.get('registered_name')}"
ENDPOINT = dbutils.widgets.get("endpoint_name")
WEIGHTS_URL = dbutils.widgets.get("weights_url")

LOCAL_WEIGHTS = "/tmp/yolo_weights.pt"
if not os.path.exists(LOCAL_WEIGHTS):
    LOG.info("Downloading YOLO weights from %s", WEIGHTS_URL)
    urllib.request.urlretrieve(WEIGHTS_URL, LOCAL_WEIGHTS)
LOG.info("Weights at %s (%d bytes)", LOCAL_WEIGHTS, os.path.getsize(LOCAL_WEIGHTS))

## PyFunc wrapper

The `predict()` method accepts a pandas DataFrame (or list-of-dicts) where
each row has `image` (base64), `conf`, and `iou`. It returns one detection
list per row.

In [ ]:
import base64
import io

import mlflow
import mlflow.pyfunc
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from PIL import Image


class YoloDetector(mlflow.pyfunc.PythonModel):
    """PyFunc wrapper around an ultralytics YOLO model.

    Inputs (per row):
      - image: base64-encoded JPEG/PNG bytes (with or without `data:...,` prefix)
      - conf:  optional confidence threshold, default 0.35
      - iou:   optional IoU threshold for NMS, default 0.5

    Output (per row): list of detections, each a dict:
      `{label, class_id, confidence, bbox: [x1, y1, x2, y2]}`
    """

    def load_context(self, context):
        from ultralytics import YOLO
        self._model = YOLO(context.artifacts["weights"])
        self._names = self._model.names

    def _run_one(self, image_b64, conf, iou):
        if not image_b64:
            return []
        if isinstance(image_b64, str) and image_b64.startswith("data:"):
            image_b64 = image_b64.split(",", 1)[1]
        img = Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")
        results = self._model.predict(
            source=np.array(img),
            conf=float(conf or 0.35),
            iou=float(iou or 0.5),
            verbose=False,
        )
        detections = []
        for r in results:
            if r.boxes is None:
                continue
            xyxy = r.boxes.xyxy.cpu().numpy()
            cls = r.boxes.cls.cpu().numpy()
            conf_arr = r.boxes.conf.cpu().numpy()
            for box, c, s in zip(xyxy, cls, conf_arr):
                x1, y1, x2, y2 = (int(v) for v in box.tolist())
                c = int(c)
                detections.append({
                    "label": self._names.get(c, str(c)),
                    "class_id": c,
                    "confidence": float(s),
                    "bbox": [x1, y1, x2, y2],
                })
        return detections

    def predict(self, context, model_input, params=None):
        if hasattr(model_input, "to_dict"):
            rows = model_input.to_dict(orient="records")
        elif isinstance(model_input, dict):
            rows = [model_input]
        else:
            rows = list(model_input)
        return [self._run_one(row.get("image"), row.get("conf"), row.get("iou")) for row in rows]

## Smoke-test locally and log to MLflow

Builds a tiny example payload to derive the signature, then logs the model
to MLflow and registers it in Unity Catalog.

In [ ]:
# 1x1 transparent PNG so the smoke test runs without external assets.
_TINY_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8/5+hHgAH"
    "ggJ/PchI7wAAAABJRU5ErkJggg=="
)

sample_input = pd.DataFrame([{"image": _TINY_PNG_B64, "conf": 0.35, "iou": 0.5}])
sample_output = [[]]
signature = infer_signature(sample_input, sample_output)

mlflow.set_registry_uri("databricks-uc")
# Catalog is assumed to already exist; only ensure the schema.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

with mlflow.start_run(run_name="yolo_deploy") as run:
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=YoloDetector(),
        artifacts={"weights": LOCAL_WEIGHTS},
        signature=signature,
        input_example=sample_input,
        registered_model_name=REGISTERED,
        pip_requirements=[
            "mlflow>=2.13",
            "ultralytics==8.3.0",
            "torch>=2.0.0",
            "torchvision>=0.15.0",
            "numpy<2",
            "pillow",
        ],
    )
LOG.info("Logged model: %s", info.model_uri)

## Create / update the Model Serving endpoint

Uses the latest registered version and a `Small` scale-to-zero serving
config. The endpoint name must match `app.yaml`'s
`DATABRICKS_SERVING_ENDPOINT_DETECTOR`.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED}'")
latest_version = max(versions, key=lambda v: int(v.version)).version
LOG.info("Deploying %s version %s to endpoint %s", REGISTERED, latest_version, ENDPOINT)

served = ServedEntityInput(
    entity_name=REGISTERED,
    entity_version=latest_version,
    workload_size="Small",
    scale_to_zero_enabled=True,
)

w = WorkspaceClient()
try:
    w.serving_endpoints.get(name=ENDPOINT)
    LOG.info("Endpoint exists, updating config")
    w.serving_endpoints.update_config(name=ENDPOINT, served_entities=[served])
except Exception:
    LOG.info("Endpoint not found, creating")
    w.serving_endpoints.create(
        name=ENDPOINT,
        config=EndpointCoreConfigInput(name=ENDPOINT, served_entities=[served]),
    )
LOG.info("Submitted deployment for endpoint %s", ENDPOINT)
LOG.info("Track progress in the Serving UI; first-time build takes ~5-10 min.")